In [8]:
import re
import os
import yaml
import wandb
with open("configs/bert_keyetm.yaml", 'r') as ymlfile:
    config = yaml.load(ymlfile, Loader=yaml.FullLoader)
config_model = config['model']
wandb.init(project="diagnosis", config=config_model)
import logging
import os.path as osp
import pandas as pd
import numpy as np
import nltk
#nltk.download('punkt')
#nltk.download('stopwords')
import torch

from embedded_topic_model.model.etm import ETM
from collections import defaultdict
from embedded_topic_model.utils import preprocessing
from torchmetrics.classification import MulticlassPrecision, MulticlassRecall, MulticlassF1Score


## Preprocessing

In [9]:
# read in dataset
test_data = pd.read_csv('Data/pain_study/test_data.csv')
test_index = list(range(len(test_data)))
lab_cols = [col for col in test_data.columns if col.startswith("label_")]
lab_cols = sorted(lab_cols)
test_labels = test_data[lab_cols].to_numpy()
print(test_index)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125]


In [12]:
%%bash
export MKL_SERVICE_FORCE_INTEL=1
python get_oov_emb.py
python get_iv_seed.py --topm 10

Error: mkl-service + Intel(R) MKL: MKL_THREADING_LAYER=INTEL is incompatible with libgomp.so.1 library.
	Try to import numpy first or set the threading layer accordingly. Set MKL_SERVICE_FORCE_INTEL to force it.
/home/zzhaozhe/miniconda3/envs/keyetm_pain/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/zzhaozhe/miniconda3/envs/keyetm_pain/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/zzhaozhe/miniconda3/envs/keyetm_pain/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect m

Loading data... 

The size of vocabulary is 2542
Constructing vocabulary... 

Loading seeds... 

create embeddings in bert space...


100%|██████████| 2590/2590 [00:24<00:00, 103.77it/s]


In [13]:
# read keywords
seedwords = preprocessing.read_seedword("Data/pain_study/keywords/keywords_bert_1.txt", stem_words=False)

In [14]:
abstracts = test_data['abstracts'].to_list()
cnt = defaultdict(int)
for abs in abstracts:
	data = re.split(r'[^\w_]+', abs)
	for word in data:
		cnt[word] += 1

min_count = 3
vocab = set()
for word in cnt:
	if cnt[word] >= min_count and word.replace('_', ' ').strip() != '':
		vocab.add(word)

print(f"The size of vocabulary for test set is {len(vocab)}")

The size of vocabulary for test set is 2589


In [5]:
print(test_index)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125]


### Original Abstracts

In [7]:
# modeling without any modification
vocabulary1, train_dataset1, test_dataset1 = preprocessing.create_etm_datasets(
                            abstracts,
                            vocab,
                            test_index=test_index,
                            test_labels=test_labels,
                            min_df=0.02,
                            max_df=0.9,
                            stem_words=False,
                            )

embeddings_file = osp.join("Data/pain_study/embeddings/embedding_bert.txt")
embeddings_mapping = {}
with open(embeddings_file) as fin:
    for line in fin:
        data = line.strip().split()
        if len(data) != 769:
            continue
        word = data[0]
        emb = np.array([float(x) for x in data[1:]])
        embeddings_mapping[word] = emb

gamma_prior,gamma_prior_bin = preprocessing.get_gamma_prior_bert(vocabulary1,seedwords,14,1,embeddings_mapping,False)
logger = logging.getLogger(__name__)
logging.basicConfig(filename="Data/pain_study/diagnosis/logs/bert_iv.log", filemode='w', level=logging.DEBUG)
etm_instance = ETM(
                   vocabulary1,
                   logger=logger,
                   batch_size = 1,
                   embeddings=embeddings_mapping,
                   num_topics=14,
                   epochs=10,
                   enc_drop = 0.0,
                   lambda_theta = 35,
                   lambda_alpha = 15,
                   theta_act = "softplus",
                   lr = 0.005,
                   gamma_prior = gamma_prior,
                   gamma_prior_bin=gamma_prior_bin,
                   rho_size=768,
                   emb_size=768,
                   t_hidden_size=64,
                   train_embeddings=False)

etm_instance.fit(train_dataset1, test_dataset1, label_threshold=0.0714)

def write_to_file(res_path,file_name,results, model_name):
    if(torch.is_tensor(results)):   
         df = pd.DataFrame(results.numpy())
    else:
         df = pd.DataFrame(results)
    if("doc_topic_dist.csv" in file_name):
         #df= df.drop(['Unnamed: 0'],axis=1)
         #labels = []
         a = df.to_numpy()
         top3_labels = np.argsort(a, axis=1)[:, -3:]
         np.savetxt(os.path.join(res_path,f'{model_name}_ETM_multi_labels_.csv'), top3_labels, fmt='%d', delimiter=',')
         #for i in range(len(a)):
         #    labels.append(np.asarray(a[i]).argmax())
         #with open(os.path.join(res_path,f'{model_name}_ETM_multi_labels_.csv'),'w') as f:
         #    for item in range(len(top3_labels)):
         #        f.write(','.join(top3_labels[item].))
         #        f.write("\n")
                 
    df.to_csv(os.path.join(res_path,file_name), index=False)

write_to_file('Data/pain_study/diagnosis/','bert_doc_topic_dist.csv',etm_instance.get_document_topic_dist(),"bert")



Building vocabulary...
Initial vocabulary size: 2589
Tokenizing documents and splitting into train/test...
vocabulary after removing words not in train: 2493
Number of documents (train_dataset): 126 [this should be equal to 126]
Number of documents (test_dataset): 126 [this should be equal to 126]
Removing empty documents...
[]
[]
len(words_train):  27767
len(words_test):  27767
len(np.unique(doc_indices_train)): 126 [this should be 126]
len(np.unique(doc_indices_test)): 126 [this should be 126]
ind_enabling not in vocabulary 

toxicity not in vocabulary 

treatment_delivery not in vocabulary 

chemotherapy not in vocabulary 

vaccines not in vocabulary 

modification not in vocabulary 

hiv not in vocabulary 

indications not in vocabulary 

survival not in vocabulary 

standardization not in vocabulary 

research_training not in vocabulary 

documentation not in vocabulary 

research_approach not in vocabulary 

research_approaches not in vocabulary 

advocacy not in vocabulary 

cli

## Modifying Data

In [5]:
 # preparation for modification
def split_into_sentences(text):
    # Simple function to split text into sentences.
    # You might need more sophisticated sentence splitting depending on the text.
    sentence_endings = re.compile(r'(?<!\w\.\w.)(?<=\.|\?)\s')
    return sentence_endings.split(text)

def find_and_append_sentences(documents, labels, keywords_list, repeat_times=5):
    updated_documents = []

    for doc, label in zip(documents, labels):
        # Get the list of keyword for this document's label
        try:
            keywords = keywords_list[label]
        except IndexError:
            updated_documents.append(doc)
            continue

        keyword_pattern = re.compile(r'\b(?:' + '|'.join(re.escape(kw) for kw in keywords) + r')\b', re.IGNORECASE)

        # Split document into sentences
        sentences = split_into_sentences(doc)
        important_sentences = []

        for sentence in sentences:
            if keyword_pattern.search(sentence):
                important_sentences.append(sentence * repeat_times)
        
        # Append important sentences to the document
        updated_document = doc + ' ' + ' '.join(important_sentences)
        updated_documents.append(updated_document.strip())

    return updated_documents

def find_and_append_keywords_without_splitting(documents, labels, keywords_list, repeat_times=5):
    updated_documents = []

    for doc, label in zip(documents, labels):
        try:
            keywords = keywords_list[label]
        except IndexError:
            updated_documents.append(doc)
            continue
        # Get the list of keywords for this document's label
        keyword_set = set(map(str.lower, keywords)) # Set for quick lookup
        keyword_pattern = re.compile(r'\b(?:' + '|'.join(re.escape(kw) for kw in keywords) + r')\b', re.IGNORECASE)

        # Find all keywords in the document
        found_keywords = keyword_pattern.findall(doc)
        
        # Clean and deduplicate the list
        unique_keywords = list(set(kw.lower() for kw in found_keywords if kw.lower() in keyword_set))

        # Repeat found keywords as specified and append at the end of the document
        keywords_text = ' '.join(unique_keywords * repeat_times).strip()
        updated_document = doc + ' ' + keywords_text if keywords_text else doc
        updated_documents.append(updated_document)

    return updated_documents

### Append Key Sentences


In [15]:
# modify data1: add key sentences
# read the original label
origin_label = pd.read_csv(os.path.join('Data/pain_study/', 'pain_grants.csv'))
test_indecies = origin_label[origin_label['primary_label'].notna()].index
test_set = origin_label.loc[test_indecies]
primary_label = test_set['primary_label'].tolist()
primary_label = [int(x-1.0) for x in primary_label]
new_abstracts = find_and_append_sentences(abstracts,primary_label,seedwords,50)
print(new_abstracts[0])

NameError: name 'find_and_append_sentences' is not defined

In [10]:
# train the model again
vocabulary2, train_dataset2, test_dataset2 = preprocessing.create_etm_datasets(
                            new_abstracts,
                            vocab,
                            test_index=test_index,
                            test_labels=test_labels,
                            min_df=0.02,
                            max_df=0.9,
                            stem_words=False,
                            )

embeddings_file = osp.join("Data/pain_study/embeddings/embedding_bert.txt")
embeddings_mapping = {}
with open(embeddings_file) as fin:
    for line in fin:
        data = line.strip().split()
        if len(data) != 769:
            continue
        word = data[0]
        emb = np.array([float(x) for x in data[1:]])
        embeddings_mapping[word] = emb

gamma_prior,gamma_prior_bin = preprocessing.get_gamma_prior_bert(vocabulary2,seedwords,14,1,embeddings_mapping,False)
logger1 = logging.getLogger(__name__)
logging.basicConfig(filename="Data/pain_study/diagnosis/logs/bert_iv_sent.log", filemode='w', level=logging.DEBUG)
etm_instance = ETM(
                   vocabulary2,
                   logger=logger1,
                   batch_size = 1,
                   embeddings=embeddings_mapping,
                   num_topics=14,
                   epochs=10,
                   enc_drop = 0.0,
                   lambda_theta = 35,
                   lambda_alpha = 15,
                   theta_act = "softplus",
                   lr = 0.005,
                   gamma_prior = gamma_prior,
                   gamma_prior_bin=gamma_prior_bin,
                   rho_size=768,
                   emb_size=768,
                   t_hidden_size=64,
                   train_embeddings=False)

etm_instance.fit(train_dataset2, test_dataset2, label_threshold=0.0714)

def write_to_file(res_path,file_name,results, model_name):
    if(torch.is_tensor(results)):   
         df = pd.DataFrame(results.numpy())
    else:
         df = pd.DataFrame(results)
    if("doc_topic_dist.csv" in file_name):
         #df= df.drop(['Unnamed: 0'],axis=1)
         #labels = []
         a = df.to_numpy()
         top3_labels = np.argsort(a, axis=1)[:, -3:]
         np.savetxt(os.path.join(res_path,f'sent_{model_name}_ETM_multi_labels_.csv'), top3_labels, fmt='%d', delimiter=',')
         #for i in range(len(a)):
         #    labels.append(np.asarray(a[i]).argmax())
         #with open(os.path.join(res_path,f'{model_name}_ETM_multi_labels_.csv'),'w') as f:
         #    for item in range(len(top3_labels)):
         #        f.write(','.join(top3_labels[item].))
         #        f.write("\n")
                 
    df.to_csv(os.path.join(res_path,file_name), index=False)

write_to_file('Data/pain_study/diagnosis/','sent_bert_doc_topic_dist.csv',etm_instance.get_document_topic_dist(),"bert")

# classification results
# read the predicted label
model_output = pd.read_csv(os.path.join('Data/pain_study/diagnosis/', 'sent_bert_ETM_labels_.csv'),names=["output"])

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print(precision_score(primary_label, model_output['output'], average='macro', zero_division=np.nan))
print(recall_score(primary_label, model_output['output'], average='macro', zero_division=np.nan))
print(f1_score(primary_label, model_output['output'], average='micro', zero_division=np.nan))
print(f1_score(primary_label, model_output['output'], average='macro', zero_division=np.nan))

primary_label_ts = torch.tensor(primary_label)
preds = pd.read_csv(os.path.join('Data/pain_study/diagnosis/', 'sent_bert_doc_topic_dist.csv'), header=0)
preds = torch.tensor(preds.values, dtype=torch.float32)
metric = MulticlassPrecision(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Precision: {metric(preds, primary_label_ts)}")
metric = MulticlassRecall(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Recall: {metric(preds, primary_label_ts)}")
metric = MulticlassF1Score(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Macro F1: {metric(preds, primary_label_ts)}")


Building vocabulary...
Initial vocabulary size: 2589
Tokenizing documents and splitting into train/test...
vocabulary after removing words not in train: 2493
Number of documents (train_dataset): 126 [this should be equal to 126]
Number of documents (test_dataset): 126 [this should be equal to 126]
Removing empty documents...
[]
[]
len(words_train):  331117
len(words_test):  331117
len(np.unique(doc_indices_train)): 126 [this should be 126]
len(np.unique(doc_indices_test)): 126 [this should be 126]
ug3
r33
100
60
180
16
48
d1n
uh3
47
s1
2
nf1
13
d2n
10
2d
70
18
r61
160
30
8
r34
65
4
3
cd22
cdk5
5
0
runx1
1b
24
80
9
6
250
15
cyp2d6
m2va
nudt21
a2ar
11
500
d3r
12
tlr4
cd163
a2cps
m2
000
t2
d2
38
20
ed2
7
m1
sirt1
r01
50
nrf2
mrgprb2
2020
k23
1
gucy2c
asct2
eh302
14
d2r
3rp
200
40
bcl6
mrgprx2
trpv1
19
vk4
0.03149407911312673
0.03223443223443223
0.031746031746031744
0.010256410256410256
Top-3 Precision: 0.27434396743774414
Top-3 Recall: 0.23219281435012817
Top-3 Macro F1: 0.166877582669258

Top-3 Precision: 0.2429986596107483
Top-3 Recall: 0.2010101079940796
Top-3 Macro F1: 0.18169499933719635

### Append Keywords

In [12]:
# modify data2: add keywords
# read the original label
origin_label = pd.read_csv(os.path.join('Data/pain_study/', 'pain_grants.csv'))
test_indecies = origin_label[origin_label['primary_label'].notna()].index
test_set = origin_label.loc[test_indecies]
primary_label = test_set['primary_label'].tolist()
primary_label = [int(x-1.0) for x in primary_label]
new_abstracts2 = find_and_append_keywords_without_splitting(abstracts,primary_label,seedwords,50)
print(new_abstracts2[78])

project summary the opioid_epidemic has encouraged the biomedical research community to explore new pain targets and further study opioids most famous target, the mu_opioid_receptor. one underexplored arena is the study of the endogenous opioid peptide system, which we know very little especially in comparison to our knowledge of the receptors they activate. this is despite the commonsense understanding that the endogenous peptides must change as a result of the allostatic load imposed by chronic_pain, opioids, and chronic opioid exposure. there are well_established sex_differences in the actions of opioids at the mu_opioid_receptor, but it is unknown whether there are also differences in the endogenous opioid peptides. the experiments outlined here lay the necessary and long overdue groundwork to understand the neural mechanisms of one class of the endogenous opioids, called enkephalins, and how they contribute to pain. in the first aim, we will quantify enkephalin mrna expression in 

In [8]:
# train the model third time
vocabulary3, train_dataset3, test_dataset3 = preprocessing.create_etm_datasets(
                            new_abstracts2,
                            vocab,
                            test_index=test_index,
                            test_labels=test_labels,
                            min_df=0.02,
                            max_df=0.9,
                            stem_words=False,
                            )

embeddings_file = osp.join("Data/pain_study/embeddings/embedding_bert.txt")
embeddings_mapping = {}
with open(embeddings_file) as fin:
    for line in fin:
        data = line.strip().split()
        if len(data) != 769:
            continue
        word = data[0]
        emb = np.array([float(x) for x in data[1:]])
        embeddings_mapping[word] = emb

gamma_prior,gamma_prior_bin = preprocessing.get_gamma_prior_bert(vocabulary3,seedwords,14,1,embeddings_mapping,False)
logger1 = logging.getLogger(__name__)
logging.basicConfig(filename="Data/pain_study/diagnosis/logs/bert_iv_keywords.log", filemode='w', level=logging.DEBUG)
etm_instance = ETM(
                   vocabulary3,
                   logger=logger1,
                   batch_size = 1,
                   embeddings=embeddings_mapping,
                   num_topics=14,
                   epochs=10,
                   enc_drop = 0.0,
                   lambda_theta = 35,
                   lambda_alpha = 15,
                   theta_act = "softplus",
                   lr = 0.005,
                   gamma_prior = gamma_prior,
                   gamma_prior_bin=gamma_prior_bin,
                   rho_size=768,
                   emb_size=768,
                   t_hidden_size=64,
                   train_embeddings=False)

etm_instance.fit(train_dataset3, test_dataset3, label_threshold=0.0714)

def write_to_file(res_path,file_name,results, model_name):
    if(torch.is_tensor(results)):   
         df = pd.DataFrame(results.numpy())
    else:
         df = pd.DataFrame(results)
    if("doc_topic_dist.csv" in file_name):
         #df= df.drop(['Unnamed: 0'],axis=1)
         #labels = []
         a = df.to_numpy()
         top3_labels = np.argsort(a, axis=1)[:, -3:]
         np.savetxt(os.path.join(res_path,f'keywords_{model_name}_ETM_multi_labels_.csv'), top3_labels, fmt='%d', delimiter=',')
         #for i in range(len(a)):
         #    labels.append(np.asarray(a[i]).argmax())
         #with open(os.path.join(res_path,f'{model_name}_ETM_multi_labels_.csv'),'w') as f:
         #    for item in range(len(top3_labels)):
         #        f.write(','.join(top3_labels[item].))
         #        f.write("\n")
                 
    df.to_csv(os.path.join(res_path,file_name), index=False)


write_to_file('Data/pain_study/diagnosis/','keywords_bert_doc_topic_dist.csv',etm_instance.get_document_topic_dist(),"bert")

# classification results
# read the predicted label
model_output = pd.read_csv(os.path.join('Data/pain_study/diagnosis/', 'keywords_bert_ETM_labels_.csv'),names=["output"])

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print(precision_score(primary_label, model_output['output'], average='macro', zero_division=np.nan))
print(recall_score(primary_label, model_output['output'], average='macro', zero_division=np.nan))
print(f1_score(primary_label, model_output['output'], average='micro', zero_division=np.nan))
print(f1_score(primary_label, model_output['output'], average='macro', zero_division=np.nan))

primary_label_ts = torch.tensor(primary_label)
preds = pd.read_csv(os.path.join('Data/pain_study/diagnosis/', 'keywords_bert_doc_topic_dist.csv'), header=0)
preds = torch.tensor(preds.values, dtype=torch.float32)
metric = MulticlassPrecision(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Precision: {metric(preds, primary_label_ts)}")
metric = MulticlassRecall(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Recall: {metric(preds, primary_label_ts)}")
metric = MulticlassF1Score(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Macro F1: {metric(preds, primary_label_ts)}")

NameError: name 'new_abstracts2' is not defined

Original Texts:
Top-3 Precision: 0.2429986596107483
Top-3 Recall: 0.2010101079940796
Top-3 Macro F1: 0.18169499933719635

Extra Sent:
Top-3 Precision: 0.27434396743774414
Top-3 Recall: 0.23219281435012817
Top-3 Macro F1: 0.16687758266925812

Extra Seeds:
Top-3 Precision: 0.7186066508293152
Top-3 Recall: 0.5569597482681274
Top-3 Macro F1: 0.529565155506134

## TF-IDF

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Step 1: Initialize a TfidfVectorizer
vectorizer = TfidfVectorizer()

# Step 2: Learn vocabulary and idf, then return term-document matrix.
tfidf_matrix = vectorizer.fit_transform(abstracts)

# Get feature names (vocabulary)
feature_names = vectorizer.get_feature_names_out()

# Function to compute sum of TF-IDF scores for each keyword list within a document
def sum_tfidf_for_keywords(doc_index, keywords, tfidf_mat, feature_names):
    scores = tfidf_mat[doc_index].toarray().flatten()
    keyword_score_sum = sum(scores[feature_names.tolist().index(kw)] for kw in keywords if kw in feature_names)
    return keyword_score_sum

# Step 3: Calculate summed TF-IDF scores for each keyword list in each document
keyword_sums = []
for doc_index in range(len(abstracts)):
    keyword_sums_for_doc = []
    for keyword_list in seedwords:
        total_tfidf = sum_tfidf_for_keywords(doc_index, keyword_list, tfidf_matrix, feature_names)
        keyword_sums_for_doc.append(total_tfidf)
    keyword_sums.append(keyword_sums_for_doc)

# Output results
for doc_idx, sums in enumerate(keyword_sums):
    print(f"Document {doc_idx + 1}: Keyword list sums: {sums}")


Document 1: Keyword list sums: [0.0, 0.0, 0.0, 0.0, 0.0, 0.15148819706644223, 0.05747848984097728, 0.0, 0.03458890154725027, 0.048070363990100985, 0.11000765809556498, 0.0, 0.0]
Document 2: Keyword list sums: [0.02956421489129009, 0.0, 0.0, 0.0, 0.0, 0.07042716734465433, 0.0, 0.0, 0.0, 0.10214625324842261, 0.0, 0.0, 0.050959615930740075]
Document 3: Keyword list sums: [0.037913506048067695, 0.06267467848574622, 0.0, 0.0, 0.10668759377638701, 0.0, 0.0, 0.06512796480149718, 0.0, 0.06675527062744659, 0.0, 0.05303040644442744, 0.07346345204026747]
Document 4: Keyword list sums: [0.0, 0.03783055695435685, 0.02548414791342249, 0.0, 0.2937136843963285, 0.0, 0.0, 0.04403168728623032, 0.0, 0.16391203710625246, 0.0, 0.0, 0.024300873882460695]
Document 5: Keyword list sums: [0.24535752478810904, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.12204391607651091, 0.0, 0.0, 0.0, 0.03261350933834618]
Document 6: Keyword list sums: [0.0, 0.0, 0.0909783358665559, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.08507468209152529,

 84%|████████▎ | 13885/16600 [01:48<00:21, 124.49it/s]

In [17]:
keyword_sums = [sums + [0] for sums in keyword_sums]

In [18]:
primary_label_ts = torch.tensor(primary_label)
preds = torch.tensor(keyword_sums, dtype=torch.float32)
metric = MulticlassPrecision(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Precision: {metric(preds, primary_label_ts)}")
metric = MulticlassRecall(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Recall: {metric(preds, primary_label_ts)}")
metric = MulticlassF1Score(num_classes=14, average='macro', top_k=3)
print(f"Top-3 Macro F1: {metric(preds, primary_label_ts)}")

Top-3 Precision: 0.43911054730415344
Top-3 Recall: 0.4196997582912445
Top-3 Macro F1: 0.3589524030685425


In [21]:
def sum_tfidf_for_keywords(doc_index, keywords, tfidf_mat, feature_names):
    scores = tfidf_mat[doc_index].toarray().flatten()
    keyword_scores = {kw: scores[feature_names.tolist().index(kw)] for kw in keywords if kw in feature_names}
    return sum(keyword_scores.values()), keyword_scores

# Calculate summed TF-IDF scores and store individual scores
all_sums_with_scores = []
for doc_index in range(len(abstracts)):
    sums_with_scores_for_doc = []
    for idx, keyword_list in enumerate(seedwords):
        total_tfidf, keyword_scores = sum_tfidf_for_keywords(doc_index, keyword_list, tfidf_matrix, feature_names)
        sums_with_scores_for_doc.append((total_tfidf, keyword_scores, idx))
    all_sums_with_scores.append(sums_with_scores_for_doc)

for doc_idx, sums_with_scores in enumerate(all_sums_with_scores):
    # Find the keyword list with the maximum sum of TF-IDF scores
    max_sum, keyword_scores, max_index = max(sums_with_scores, key=lambda x: x[0])
    print(f"Document {doc_idx + 1}: Highest TF-IDF keyword list sum: {max_index}, primary label: {primary_label[doc_idx]}")
    print("TF-IDF scores for keywords in this list:")
    for keyword, score in keyword_scores.items():
        print(f"Keyword: {keyword}, Score: {score}")
    print("")

    # confusion matrix 
    

Document 1: Highest TF-IDF keyword list sum: 5, primary label: 7
TF-IDF scores for keywords in this list:
Keyword: plasticity, Score: 0.04950042822295453
Keyword: currents, Score: 0.0
Keyword: sensitization, Score: 0.0
Keyword: nerve_blocks, Score: 0.0
Keyword: inhibitory, Score: 0.05747848984097728
Keyword: neuron_specific, Score: 0.0
Keyword: neuronal, Score: 0.04450927900251042

Document 2: Highest TF-IDF keyword list sum: 9, primary label: 5
TF-IDF scores for keywords in this list:
Keyword: self_management, Score: 0.0
Keyword: therapy, Score: 0.031719085903768277
Keyword: adherence, Score: 0.0
Keyword: complementary, Score: 0.0
Keyword: behavior, Score: 0.0
Keyword: process, Score: 0.0
Keyword: recovery, Score: 0.0
Keyword: inhibition, Score: 0.0
Keyword: expertise, Score: 0.0
Keyword: pathways, Score: 0.07042716734465433

Document 3: Highest TF-IDF keyword list sum: 4, primary label: 4
TF-IDF scores for keywords in this list:
Keyword: prospective, Score: 0.0
Keyword: transition, S